# RAG System with RAGBench - Using Ingestion Layer and Strategy Classes

This notebook demonstrates the complete RAG pipeline using:
- Ingestion Layer for loading and parsing RAGBench datasets
- Strategy Factory for creating strategies
- RAG Config for configuration
- Complete RAG Pipeline for orchestration

## 1. Setup & Dependencies

In [1]:
!pip install datasets faiss-cpu sentence-transformers torch groq python-dotenv nltk -q

zsh:1: /Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/venv/bin/pip: bad interpreter: /Users/adityanarayan/aiml-talentsprint/rag-capstone-project/real-world-rag-system/venv/bin/python: no such file or directory

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /Users/adityanarayan/.pyenv/versions/3.11.9/bin/python -m pip install --upgrade pip


## 2. Imports

In [2]:
import os
import json
import pandas as pd
from typing import List, Dict, Any
from dotenv import load_dotenv
from groq import Groq
import numpy as np

# Load environment variables
load_dotenv(override=True)

# Get API keys
HF_TOKEN = os.getenv("HF_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print(f"HuggingFace token loaded: {bool(HF_TOKEN)}")
print(f"Groq API key loaded: {bool(GROQ_API_KEY)}")

HuggingFace token loaded: True
Groq API key loaded: True


## 3. Load Configuration

In [ ]:
from rag.config.loader import ConfigLoader

# Load the high quality config (with reranking) from its YAML file.
# Providers are built automatically from `config.providers` when the
# pipeline is created, resolving credentials from each provider's `api_key_env`.
CONFIG_PATH = "config/rag_config_high_quality.yaml"
config = ConfigLoader.load(CONFIG_PATH)

print("Configuration:")
print(f"  Chunking:   {config.chunking.type.value}")
print(f"  Embedding:  {config.embedding.type.value} ({config.embedding.config.model_name})")
print(f"  Retrieval:  {config.retrieval.type.value}")
print(f"  Reranker:   {config.reranker.type.value} ({config.reranker.config.model_name})")
print(f"  Generation: {config.generation.strategy.value} (provider={config.generation.provider}, model={config.generation.config.model})")
print(f"  Evaluation: {config.evaluation.type.value} (provider={config.evaluation.provider})")

📋 Configuration:
  Chunking: sentence
  Embedding: sentence_transformer (BAAI/bge-large-en-v1.5D)
  Retrieval: dense_rerank
  Reranker: cross_encoder (BAAI/bge-reranker-v2-m3D)
  Generation: groq (llama-3.3-70b-versatile)
  Evaluation: trace


## 4. Load Data Using Data Loader

In [8]:
## 4. Load Data Using Ingestion Layer
from ingestion import (
    RAGBenchCovidQALoader,
    ParserFactory,
    ParserType,
    DataProcessor,
    DatasetLoadingConfig
)

# Create loader for CovidQA dataset
dataset_loading_config = DatasetLoadingConfig(limit=246)  # Full dataset
loader = RAGBenchCovidQALoader(
    split="test",
    config=dataset_loading_config
)

# Load raw data
print("📥 Loading RAGBench CovidQA dataset...")
raw_data = loader.load()
print(f"✅ Loaded {len(raw_data)} samples\n")

# Get dataset info
info = loader.info()
print(f"Dataset Info:")
print(f"  Source: {info['source']}")
print(f"  Dataset: {info['dataset_name']}")
print(f"  Split: {info['split']}")
print(f"  Samples: {info['num_samples']}")
print(f"  Keys: {len(info['keys'])} fields\n")

# Inspect first sample
first_sample = loader.load_sample(0)
print(f"First Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")
print(f"  Has ground truth scores: {bool(first_sample.get('relevance_score'))}")

/Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Loading RAGBench CovidQA dataset...
Loading RAGBench dataset: covidqa (test)...
Loaded 246 samples
✅ Loaded 246 samples

Dataset Info:
  Source: galileo-ai/ragbench
  Dataset: covidqa
  Split: test
  Samples: 246
  Keys: 26 fields

First Sample:
  Question: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?...
  Documents: 4
  Has ground truth scores: True


## 5. Parse Documents

In [9]:
## 5. Parse Documents Using Ingestion Layer
# Create parser strategy
print("📄 Creating parser strategy...")
parser = ParserFactory.create_parser(ParserType.TITLE_PASSAGE)
print(f"✅ Parser created: {parser.__class__.__name__}\n")

# Create data processor with parser strategy
processor = DataProcessor(parser_strategy=parser)

# Process dataset into canonical Document objects
print("📄 Processing documents...")
documents = processor.process_dataset(raw_data)
print(f"✅ Processed {len(documents)} documents\n")

# Print sample
sample_doc = documents[0]
print(f"Sample RAG Document:")
print(f"  Title: {sample_doc.title[:60]}...")
print(f"  Content: {sample_doc.content[:100]}...")
print(f"  Metadata: {sample_doc.metadata}")

📄 Creating parser strategy...
✅ Parser created: TitlePassageParser

📄 Processing documents...
✅ Processed 984 documents

Sample RAG Document:
  Title: Type I Interferon Receptor Deficiency in Dendritic Cells Fac...
  Content: successful treatment for HCV serves to circumvent the viral inhibition of IFN induction. Thus, HCV m...
  Metadata: {'doc_id': 'sample_0_doc_0', 'sample_index': 0, 'source': 'ragbench', 'parser_type': 'title_passage'}


## 6. Models Are Config-Driven

Both the embedding model and the reranking model are created inside the pipeline from the RAG config (`config.embedding` / `config.reranker`). No manual model loading is required, and no API clients are passed in: providers are built automatically from `config.providers`, resolving credentials from each provider's `api_key_env`.

In [10]:
# Both the embedding model and the reranking model are created inside the
# pipeline from the RAG config (config.embedding / config.reranker).
# No manual model loading or API clients are needed here -- providers are
# built automatically from config.providers when the pipeline is created.
print("Embedding + reranking models will be built by the pipeline from config.")

Embedding + reranking models will be built by the pipeline from config.


## 7. Initialize RAG Pipeline

In [ ]:
from rag.pipeline.rag_pipeline import RAGPipeline

# The pipeline is a pure orchestrator. It builds every strategy from the config
# object and registers providers automatically from config.providers -- no
# strategy-specific parameters or API clients are passed in here.
print("Initializing RAG Pipeline...")
pipeline = RAGPipeline(config)
print("Pipeline initialized\n")

# Verify strategies were created
print("Strategies created:")
print(f"  Chunker:   {pipeline.chunker.__class__.__name__}")
print(f"  Embedder:  {pipeline.embedder.__class__.__name__}")
print(f"  Reranker:  {pipeline.reranker.__class__.__name__}")
print(f"  Retriever: {pipeline.retriever.__class__.__name__}")
print(f"  Generator: {pipeline.generator.__class__.__name__}")
print(f"  Evaluator: {pipeline.evaluator.__class__.__name__}")

🔧 Initializing RAG Pipeline...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7880.83it/s]


✅ Pipeline initialized

Strategies created:
  Chunker: SentenceChunkingStrategy
  Embedder: SentenceTransformerEmbeddingStrategy
  Reranker: CrossEncoderRerankerStrategy
  Retriever: DenseRerankRetrievalStrategy
  Generator: GroqGenerationStrategy
  Evaluator: TRACeEvaluationStrategy


## 8. Build Vector Index

In [12]:
# Build index - this will chunk, embed, and store
print("🏗️  Building vector index...")
pipeline.build_index(documents)
print(f"✅ Index built and ready for querying")

🏗️  Building vector index...
Processing 984 documents...
Created 984 chunks
Generating embeddings for 984 chunks...
Vector store ready with 984 chunks
✅ Index built and ready for querying


## 9. Test Single Query

In [13]:
# Test with first sample
test_sample = raw_data[0]
test_query = test_sample['question']

print(f"❓ Test Query:")
print(f"   {test_query}\n")

# Run query through pipeline
result = pipeline.query(test_query)

# Display results
print(f"\n📊 Query Results:")
print(f"\nRetrieved Documents: {len(result['retrieved_docs'])}")
for i, doc in enumerate(result['retrieved_docs'][:3], 1):
    print(f"  {i}. {doc['metadata']['title'][:60]}...")

print(f"\n💬 Generated Response:")
print(result['response'][:300] + "...\n")

print(f"📈 TRACe Scores:")
scores = result['scores']
print(f"  Relevance:    {scores['relevance_score']:.4f}")
print(f"  Utilization:  {scores['utilization_score']:.4f}")
print(f"  Completeness: {scores['completeness_score']:.4f}")
print(f"  Adherence:    {scores['adherence_score']}")

❓ Test Query:
   Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?


Query: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?

[1] Retrieving documents...
Retrieved 5 documents

[2] Generating response...
Response generated (901 chars)

[3] Evaluating response...
Evaluation complete

📊 Query Results:

Retrieved Documents: 5
  1. Type I Interferon Receptor Deficiency in Dendritic Cells Fac...
  2. Depletion of Alveolar Macrophages Does Not Prevent Hantaviru...
  3. Confounding roles for type I interferons during bacterial an...

💬 Generated Response:
Based on the provided context, the viruses that may not cause prolonged inflammation due to strong induction of antiviral clearance are:

1. Hantavirus (as the host has time to mount a mature immune response)
2. TBFV (as it triggers various antiviral systems, including type I and III interferon, and...

📈 TRACe Scores:
  Relevance:    0.5217
  

## 10. Batch Evaluation

In [14]:
# Evaluate on batch of samples
print("🔄 Running batch evaluation on first 10 samples...\n")

evaluation_results = []

for idx in range(min(10, len(raw_data))):
    sample = raw_data[idx]
    query = sample['question']
    
    # Get ground truth
    gt_relevance = sample.get('relevance_score', 0)
    gt_utilization = sample.get('utilization_score', 0)
    gt_completeness = sample.get('completeness_score', 0)
    gt_adherence = sample.get('adherence_score', False)
    
    # Run pipeline
    try:
        result = pipeline.query(query)
        scores = result['scores']
        
        evaluation_results.append({
            'Sample': idx,
            'Query': query[:50],
            'Our Relevance': scores['relevance_score'],
            'GT Relevance': gt_relevance,
            'Our Utilization': scores['utilization_score'],
            'GT Utilization': gt_utilization,
            'Our Completeness': scores['completeness_score'],
            'GT Completeness': gt_completeness,
            'Our Adherence': scores['adherence_score'],
            'GT Adherence': gt_adherence
        })
        print(f"✅ Sample {idx}: Relevance={scores['relevance_score']:.4f} | Adherence={scores['adherence_score']}")
    except Exception as e:
        print(f"⚠️  Sample {idx} failed: {str(e)[:50]}")

# Create results dataframe
df_results = pd.DataFrame(evaluation_results)
print(f"\n✅ Completed evaluation on {len(df_results)} samples")

🔄 Running batch evaluation on first 10 samples...


Query: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?

[1] Retrieving documents...
Retrieved 5 documents

[2] Generating response...
Response generated (969 chars)

[3] Evaluating response...
Evaluation complete
✅ Sample 0: Relevance=0.3913 | Adherence=False

Query: When was the first case of COVID-19 identified?

[1] Retrieving documents...
Retrieved 5 documents

[2] Generating response...
Response generated (216 chars)

[3] Evaluating response...
Evaluation complete
✅ Sample 1: Relevance=0.1071 | Adherence=False

Query: How many antigens could be detected by Liew's multiplex ELISA test?

[1] Retrieving documents...
Retrieved 5 documents

[2] Generating response...
Response generated (130 chars)

[3] Evaluating response...
Evaluation complete
✅ Sample 2: Relevance=0.1176 | Adherence=True

Query: What is the structure of Hantaan virus?

[1] Retrieving documents...
Retrieved 5 documen

## 11. Analysis of Results

In [15]:
# Compare our scores with ground truth
print("📊 Comparison with Ground Truth\n")

print("Relevance Score:")
print(f"  Our avg:  {df_results['Our Relevance'].mean():.4f}")
print(f"  GT avg:   {df_results['GT Relevance'].mean():.4f}")
print(f"  Diff:     {abs(df_results['Our Relevance'].mean() - df_results['GT Relevance'].mean()):.4f}")

print("\nUtilization Score:")
print(f"  Our avg:  {df_results['Our Utilization'].mean():.4f}")
print(f"  GT avg:   {df_results['GT Utilization'].mean():.4f}")
print(f"  Diff:     {abs(df_results['Our Utilization'].mean() - df_results['GT Utilization'].mean()):.4f}")

print("\nCompleteness Score:")
print(f"  Our avg:  {df_results['Our Completeness'].mean():.4f}")
print(f"  GT avg:   {df_results['GT Completeness'].mean():.4f}")
print(f"  Diff:     {abs(df_results['Our Completeness'].mean() - df_results['GT Completeness'].mean()):.4f}")

print("\nAdherence (Exact Match):")
our_adherence = (df_results['Our Adherence'] == df_results['GT Adherence']).sum() / len(df_results)
print(f"  Match rate: {our_adherence:.2%}")

📊 Comparison with Ground Truth

Relevance Score:
  Our avg:  0.2162
  GT avg:   0.3044
  Diff:     0.0882

Utilization Score:
  Our avg:  0.1475
  GT avg:   0.1054
  Diff:     0.0421

Completeness Score:
  Our avg:  0.7183
  GT avg:   0.5314
  Diff:     0.1869

Adherence (Exact Match):
  Match rate: 80.00%


## 12. Interactive Querying

In [16]:
# Try custom queries
custom_queries = [
    "What is COVID-19?",
    "How is the virus transmitted?",
    "What are the symptoms?"
]

print("🔍 Custom Query Examples\n")

for query in custom_queries:
    print(f"❓ Query: {query}")
    result = pipeline.query(query)
    print(f"💬 Answer: {result['response'][:200]}...\n")

🔍 Custom Query Examples

❓ Query: What is COVID-19?

Query: What is COVID-19?

[1] Retrieving documents...
Retrieved 3 documents

[2] Generating response...
Response generated (197 chars)

[3] Evaluating response...
Evaluation complete
💬 Answer: COVID-19 is a disease caused by a virus that infects people and spreads easily from person-to-person, characterized by symptoms such as fever and cough, and has been declared a pandemic by the WHO....

❓ Query: How is the virus transmitted?

Query: How is the virus transmitted?

[1] Retrieving documents...
Retrieved 3 documents

[2] Generating response...
Response generated (437 chars)

[3] Evaluating response...
Evaluation complete
💬 Answer: The virus is transmitted mainly via direct contact or through a fomite, typically with inoculation into the eye or nose from the fingertip. Additionally, transmission by large particle aerosols has al...

❓ Query: What are the symptoms?

Query: What are the symptoms?

[1] Retrieving documents...
Retrieved 

## 13. Display Full Results Table

In [17]:
# Show full results
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

print("Full Evaluation Results:")
display(df_results[[
    'Sample', 'Query',
    'Our Relevance', 'GT Relevance',
    'Our Utilization', 'GT Utilization',
    'Our Completeness', 'GT Completeness',
    'Our Adherence', 'GT Adherence'
]])

Full Evaluation Results:


,Sample,Query,Our Relevance,GT Relevance,Our Utilization,GT Utilization,Our Completeness,GT Completeness,Our Adherence,GT Adherence
0,0,Which viruses may not cause prolonged inflamma...,0.3913,0.411765,0.3913,0.176471,1.0000,0.428571,False,False
1,1,When was the first case of COVID-19 identified?,0.1071,0.269231,0.0357,0.076923,0.3333,0.285714,False,True
2,2,How many antigens could be detected by Liew's ...,0.1176,0.125000,0.1176,0.062500,1.0000,0.500000,True,True
3,3,What is the structure of Hantaan virus?,0.4118,0.300000,0.3529,0.300000,0.8571,1.000000,True,True
4,4,How many people did SARS-CoV infect?,0.2500,0.066667,0.1500,0.066667,0.6000,1.000000,False,True
5,5,What was the focus of the study?,0.1667,0.294118,0.1667,0.176471,1.0000,0.600000,True,True
6,6,What contributed to a large part of mammalian ...,0.0476,0.055556,0.0476,0.055556,1.0000,1.000000,True,True
7,7,What is severe MARS noted for?,0.1379,0.363636,0.0345,0.045455,0.2500,0.125000,True,True
8,8,What animal models exist for both the asymptom...,0.4118,0.157895,0.0588,0.052632,0.1429,0.333333,True,True
9,9,What does Clade A contain?,0.1200,1.000000,0.1200,0.041667,1.0000,0.041667,True,True


## 14. Summary

In [18]:
print("""
╔════════════════════════════════════════════════════════╗
║           RAG SYSTEM SUMMARY                            ║
╚════════════════════════════════════════════════════════╝

✅ Components Used:
   • Ingestion Layer - Loaded and parsed RAGBench dataset
   • RAGConfig - Configuration management
   • StrategyFactory - Created all strategies
   • RAGPipeline - Orchestrated the pipeline

📊 Pipeline Configuration:
""")

print(f"   Chunking: {config.chunking.type.value} ({config.chunking.config.max_words} words)")
print(f"   Embedding: {config.embedding.type.value} ({config.embedding.config.dimension}D)")
print(f"   Retrieval: {config.retrieval.type.value} (initial_k={config.retrieval.config.initial_k})")
print(f"   Generation: {config.generation.strategy.value} ({config.generation.config.model})")
print(f"   Evaluation: {config.evaluation.type.value}")

print(f"""
📈 Results on {len(df_results)} samples:
   Relevance:    {df_results['Our Relevance'].mean():.4f} (GT: {df_results['GT Relevance'].mean():.4f})
   Utilization:  {df_results['Our Utilization'].mean():.4f} (GT: {df_results['GT Utilization'].mean():.4f})
   Completeness: {df_results['Our Completeness'].mean():.4f} (GT: {df_results['GT Completeness'].mean():.4f})
   Adherence:    {our_adherence:.2%} match rate

✨ Key Advantages:
   ✓ Clean, modular code
   ✓ Reusable components
   ✓ Configuration-driven
   ✓ Easy to extend
   ✓ Production-ready
""")


╔════════════════════════════════════════════════════════╗
║           RAG SYSTEM SUMMARY                            ║
╚════════════════════════════════════════════════════════╝

✅ Components Used:
   • Ingestion Layer - Loaded and parsed RAGBench dataset
   • RAGConfig - Configuration management
   • StrategyFactory - Created all strategies
   • RAGPipeline - Orchestrated the pipeline

📊 Pipeline Configuration:

   Chunking: sentence (100 words)
   Embedding: sentence_transformer (1024D)
   Retrieval: dense_rerank (initial_k=20)
   Generation: groq (llama-3.3-70b-versatile)
   Evaluation: trace

📈 Results on 10 samples:
   Relevance:    0.2162 (GT: 0.3044)
   Utilization:  0.1475 (GT: 0.1054)
   Completeness: 0.7183 (GT: 0.5314)
   Adherence:    80.00% match rate

✨ Key Advantages:
   ✓ Clean, modular code
   ✓ Reusable components
   ✓ Configuration-driven
   ✓ Easy to extend
   ✓ Production-ready



## 15. Generate Detailed Report with ReportGenerator

The `ReportGenerator` is a thin orchestrator over pluggable `ReportStrategy`
implementations. We use the bundled `DetailedQueryReportStrategy`, which builds:

- a **per-query table**: query, retrieved documents as an array **with all
  retrieval scores**, the generated answer, and every TRACe score next to its
  dataset ground truth plus the per-query deviation; and
- an **aggregate table**: per TRACe metric, the mean score, mean ground truth,
  standard deviation of each, and mean absolute error vs. ground truth.

It reuses the `pipeline`, `config`, and `raw_data` already built above. New
report formats can be added by implementing `ReportStrategy` and registering it
with the generator — no change to `ReportGenerator` itself.

In [ ]:
from rag.reporting import (
    DetailedQueryReportStrategy,
    PipelineRunResult,
    QueryRecord,
    ReportGenerator,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

# Number of queries to include in the report.
NUM_REPORT_QUERIES = 5

records = []
for sample in raw_data[:NUM_REPORT_QUERIES]:
    query = sample["question"]
    result = pipeline.query(query)  # retrieved_docs carry all retrieval scores
    records.append(
        QueryRecord(
            query=query,
            retrieved_docs=result["retrieved_docs"],
            answer=result["response"],
            predicted_scores=result["scores"],
            ground_truth_scores={
                "relevance_score": sample.get("relevance_score", 0.0),
                "utilization_score": sample.get("utilization_score", 0.0),
                "completeness_score": sample.get("completeness_score", 0.0),
                "adherence_score": sample.get("adherence_score", False),
            },
        )
    )

embed_cfg = config.embedding.config
run = PipelineRunResult(
    config_name="high_quality",
    records=records,
    config_summary={
        "chunking": config.chunking.type.value,
        "embedding": getattr(embed_cfg, "model_name", None) or getattr(embed_cfg, "model", None),
        "retrieval": config.retrieval.type.value,
        "gen_model": config.generation.config.model,
    },
)

generator = ReportGenerator(DetailedQueryReportStrategy())
print("Available report strategies:", generator.strategies)

report = generator.generate(run)
report.display()